# Raw sources → verified model inputs
Download hash-pinned public sources for all three sports. Frozen NBA game splits are committed metadata; no prepared outcomes or private research folders are needed. The output arrays are checked against the original study before training. Existing raw files are hash-checked, never silently substituted. Existing staged inputs must match exactly.


In [1]:
from pathlib import Path
import json, subprocess, sys, shutil
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'code/train.py').is_file(), 'Open from repository root or notebooks/'
print('Repository:', ROOT.name)
def run_command(args, cwd=None, check=True):
    """Stream subprocess output into the notebook, preserving failures."""
    proc = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.replace(str(ROOT), '.'), end='', flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f'Command failed with exit code {code}')



Repository: GM-CDLD


In [2]:
BUILD=ROOT/'work/raw_build'
BUILD.mkdir(parents=True,exist_ok=True)


In [3]:
for domain in ['NBA','beach']:
    run_command([sys.executable,str(ROOT/'tools/download_sources.py'),'--domain',domain,'--output',str(ROOT/'work/raw'/domain)])
run_command([sys.executable,str(ROOT/'tools/download_tennis.py'),'--output',str(ROOT/'work/raw/tennis')])


pbpstats_2023.tar.xz verified cached file


cdnnba_2023.tar.xz verified cached file


player_boxscores_2024.parquet verified cached file


bvb_matches_2005.csv verified cached file


bvb_matches_2006.csv verified cached file


...

bvb_matches_2008.csv verified cached file


bvb_matches_2009.csv verified cached file


atp_matches_doubles_2017.csv 452497 SHA256 OK


atp_matches_doubles_2018.csv 442785 SHA256 OK


atp_matches_doubles_2019.csv 461811 SHA256 OK


In [4]:
for script,raw,out in [('prepare_nba.py','NBA','NBA'),('prepare_tennis.py','tennis','tennis'),('prepare_beach.py','beach','beach')]:
    run_command([sys.executable,str(ROOT/'tools'/script),'--raw',str(ROOT/'work/raw'/raw),'--output',str(BUILD/out)])
run_command([sys.executable,str(ROOT/'tools/verify_inputs.py'),'--build',str(BUILD)])


{


  "domain": "NBA",


  "players": 425,


  "slots": 5,


  "context_dim": 8,


...

beach/attack_success/data/source/test.npz 8 arrays verified


beach/attack_success/data/reuse/fit.npz 8 arrays verified


beach/attack_success/data/reuse/val.npz 8 arrays verified


beach/attack_success/data/reuse/test.npz 8 arrays verified


PASS: 186 arrays match the original study inputs


In [5]:
import numpy as np
def stage_verified(source,destination):
    if not destination.exists():
        shutil.copytree(source,destination)
    else:
        for p in source.glob('*.npz'):
            target=destination/p.name
            if not target.exists():
                shutil.copy2(p,target)
                continue
            with np.load(p) as a,np.load(target) as b:
                assert set(a.files)==set(b.files)
                for key in a.files:
                    assert np.array_equal(a[key],b[key],equal_nan=True) if a[key].dtype.kind in 'fc' else np.array_equal(a[key],b[key])
        assert json.loads((source/'meta.json').read_text())==json.loads((destination/'meta.json').read_text())
    print('Staged or verified:',destination.relative_to(ROOT))
for domain in ['NBA','tennis']:
    stage_verified(BUILD/domain/'data'/domain,ROOT/'work/prepared/data'/domain)
for kind in ['source','reuse']:
    stage_verified(BUILD/'beach/attack_success/data'/kind,ROOT/'work/beach_prepared/data'/kind)


Staged or verified: work/prepared/data/NBA
Staged or verified: work/prepared/data/tennis
Staged or verified: work/beach_prepared/data/source
Staged or verified: work/beach_prepared/data/reuse
